<a href="https://colab.research.google.com/github/hemanthvnp/Linear-Discriminant-Analysis/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MvDA on Colab

Run Multi-view Discriminant Analysis on Colab, reading the **ColorFERET** images directly from your Google Drive (no manual download).

If you upload a zip of the repo instead of using GitHub, skip cell 1 and `cd` into the unzipped folder.

## 1. Get the code

In [ ]:
!git clone https://github.com/hemanthvnp/Linear-Discriminant-Analysis.git mvda
%cd mvda
!pip -q install -r requirements.txt

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Locate the ColorFERET folder and confirm it has images

If the image count is 0, your Drive copy only has metadata and you'll need the real `.ppm`/`.ppm.bz2` images.

In [ ]:
!find /content/drive/MyDrive -maxdepth 4 -iname '*colorferet*' -type d 2>/dev/null | head
FERET_ROOT = '/content/drive/MyDrive/colorferet'  # <-- set to the path printed above
!echo 'image files found:'; find "$FERET_ROOT" -type f \( -iname '*.ppm' -o -iname '*.ppm.bz2' -o -iname '*.png' -o -iname '*.jpg' \) 2>/dev/null | wc -l

## 4. Sanity check on UCI Multiple Features (auto-downloads, no Drive needed)

In [ ]:
!python experiments/run_mvda.py --dataset mfeat --mode concat --classifier ensemble

## 5. Cross-pose face recognition on ColorFERET (MvDA)

Each pose is a view, each subject is a class. `run_feret.py` reduces each pose with PCA (eigenfaces), learns a shared MvDA subspace, and classifies held-out single-pose images by nearest class mean.

`--feret-poses` picks which poses become views; subjects missing **any** chosen pose are dropped, so more poses ⇒ fewer subjects. `fa`/`fb` are the most populated. The first run reads + caches images (slow); later runs reuse the cache.

In [ ]:
!python experiments/run_feret.py \
    --feret-root "$FERET_ROOT" --feret-poses fa fb hl hr --feret-size 64 64 \
    --pca 120 --feret-max-subjects 200 --save feret_mvda.json

## 6. (optional) Frontal-only views (more subjects) and a component sweep

Using just the two frontal poses keeps the most subjects. Drop `--feret-max-subjects` to use everyone.

In [ ]:
!python experiments/run_feret.py \
    --feret-root "$FERET_ROOT" --feret-poses fa fb --feret-size 64 64 --pca 120